# TruthLens AI — Dataset Inspection & Discovery
This notebook performs systematic data discovery and inspection across all three raw datasets:
1. **FEVER** (Fact Extraction and VERification)
2. **LIAR** (PolitiFact Political Fact-Checking Benchmark)
3. **SciFact** (Scientific Claims & Rationales)

### Key Objectives:
- Confirm file formats, sizes, and record counts
- Inspect columns, schemas, and data types
- Analyze missing values and malformed records
- Verify evidence representation mechanisms

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np

# Add src to python path
sys.path.insert(0, os.path.abspath("../src"))

from fever_loader import load_fever_raw, get_fever_statistics
from liar_loader import load_all_liar_splits, get_liar_statistics, LIAR_COLUMNS
from scifact_loader import load_scifact_corpus, load_scifact_claims, reconstruct_scifact_triples, get_scifact_statistics

RAW_DIR = os.path.abspath("../data/raw")
print(f"Loading raw datasets from: {RAW_DIR}")

## 1. Load FEVER Dataset
FEVER is stored in JSON Lines format containing claim assertions labeled as `SUPPORTS`, `REFUTES`, or `NOT ENOUGH INFO`.

In [ ]:
fever_path = os.path.join(RAW_DIR, "fever", "train.jsonl")
df_fever, malformed_fever = load_fever_raw(fever_path)
print(f"FEVER Train: {len(df_fever):,} rows parsed (Malformed: {malformed_fever})")
print("Columns:", list(df_fever.columns))
display(df_fever.head(3))

## 2. Load LIAR Dataset
LIAR contains political statements rated on PolitiFact's 6-point Truth-O-Meter scale, accompanied by speaker and venue metadata.

In [ ]:
liar_dir = os.path.join(RAW_DIR, "liar")
liar_splits = load_all_liar_splits(liar_dir)

for split, df_s in liar_splits.items():
    print(f"LIAR {split.upper()}: {len(df_s):,} rows, {len(df_s.columns)} columns")

df_liar_train = liar_splits["train"]
display(df_liar_train.head(3))

## 3. Load SciFact Dataset
SciFact contains expert scientific claims paired with PubMed research paper abstracts and annotated rationale sentence IDs.

In [ ]:
scifact_dir = os.path.join(RAW_DIR, "scifact")
corpus = load_scifact_corpus(os.path.join(scifact_dir, "corpus.jsonl"))
scifact_train_claims = load_scifact_claims(os.path.join(scifact_dir, "claims_train.jsonl"))
scifact_dev_claims = load_scifact_claims(os.path.join(scifact_dir, "claims_dev.jsonl"))
scifact_test_claims = load_scifact_claims(os.path.join(scifact_dir, "claims_test.jsonl"))

print(f"SciFact Corpus: {len(corpus):,} research paper abstracts")
print(f"SciFact Claims - Train: {len(scifact_train_claims)}, Dev: {len(scifact_dev_claims)}, Test: {len(scifact_test_claims)}")

## 4. Reconstruct Grounded Triples for SciFact
Using the document IDs and sentence indices, we reconstruct explicit `(claim, evidence_text, label)` triples.

In [ ]:
scifact_triples = reconstruct_scifact_triples(scifact_train_claims, corpus, include_nei=True)
print(f"SciFact Reconstructed Triples: {len(scifact_triples):,}")
display(scifact_triples.head(5))

## 5. Dataset Summary Matrix
Comparison of key attributes across all three raw datasets.

In [ ]:
summary_data = [
    {"Dataset": "FEVER", "Domain": "General / Wikipedia", "Examples": len(df_fever), "Evidence Included": "No (Title + Sent IDs only)", "Labels": "SUPPORTS, REFUTES, NOT ENOUGH INFO (3 classes)"},
    {"Dataset": "LIAR", "Domain": "Political Speech / PolitiFact", "Examples": sum(len(d) for d in liar_splits.values()), "Evidence Included": "No (Metadata only: speaker, party, context)", "Labels": "6-point scale (pants-fire to true)"},
    {"Dataset": "SciFact", "Domain": "Biomedical / Scientific", "Examples": f"{len(scifact_train_claims)} train claims / 5,183 corpus docs", "Evidence Included": "Yes (Abstract sentences in corpus.jsonl)", "Labels": "SUPPORT, CONTRADICT, NOT ENOUGH INFO"},
]
summary_df = pd.DataFrame(summary_data)
display(summary_df)